# VigIA — treino do modelo no Colab

Modelo YOLO de pontos-chave: para cada carrinho, uma **caixa** e **3 pontos** — traseira, meio e frente.

Antes de abrir este notebook, no seu PC:

```powershell
python -m ferramentas.preparar_imagens frames_dataset_novo
python -m ferramentas.rotular
python -m ferramentas.dividir_dataset
Compress-Archive -Path dados\yolo\* -DestinationPath vigia_dataset.zip -Force
```

No Colab: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**.

In [ ]:
!pip install -q ultralytics
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA - troque o ambiente para GPU')

## 1. Enviar o `vigia_dataset.zip` com as fotos rotuladas

In [ ]:
from google.colab import files
enviado = files.upload()  # escolha vigia_dataset.zip
!rm -rf /content/yolo && mkdir -p /content/yolo
!unzip -q -o vigia_dataset.zip -d /content/yolo
!ls /content/yolo; echo; echo treino: $(ls /content/yolo/images/train | wc -l) imagens; echo val: $(ls /content/yolo/images/val | wc -l) imagens

# confere se os rótulos já estão no formato novo (caixa + 3 pontos = 14 valores por linha)
import glob
linhas = [l.split() for f in glob.glob('/content/yolo/labels/*/*.txt') for l in open(f) if l.strip()]
tamanhos = {len(l) for l in linhas}
print('carros rotulados:', len(linhas), '| valores por linha:', tamanhos)
assert tamanhos <= {14}, 'Há rótulos no formato antigo (2 pontos). Rotule de novo com traseira, meio e frente.'

## 2. Treinar

In [ ]:
from pathlib import Path
Path('/content/yolo/vigia.yaml').write_text('''path: /content/yolo
train: images/train
val: images/val
kpt_shape: [3, 3]
flip_idx: [0, 1, 2]
names:
  0: carro
''')

from ultralytics import YOLO
modelo = YOLO('yolo11n-pose.pt')
modelo.train(data='/content/yolo/vigia.yaml', epochs=100, imgsz=640, batch=16,
             project='/content/runs', name='vigia', exist_ok=True, patience=30,
             degrees=180.0, flipud=0.5, fliplr=0.5)
print('melhor modelo:', modelo.trainer.best)

## 3. Quanto o modelo está acertando

Métricas calculadas nas imagens de **validação**, que o modelo não viu durante o treino:

| Métrica | O que significa |
|---|---|
| Precisão | Das caixas que ele marcou, quantas eram carro de verdade |
| Recall | Dos carros que existiam, quantos ele encontrou |
| mAP50 caixa | Acerto geral da caixa (100% = perfeito) |
| mAP50 pontos | Acerto da traseira, meio e frente (100% = perfeito) |
| mAP50-95 pontos | Mesma ideia, exigindo pontos mais precisos |

In [ ]:
melhor = YOLO('/content/runs/vigia/weights/best.pt')
m = melhor.val(data='/content/yolo/vigia.yaml', imgsz=640, project='/content/runs', name='val', exist_ok=True)
print()
print('=== Quanto o modelo está acertando (validação) ===')
print(f'Precisão        : {m.box.mp:.1%}')
print(f'Recall          : {m.box.mr:.1%}')
print(f'mAP50 caixa     : {m.box.map50:.1%}')
print(f'mAP50 pontos    : {m.pose.map50:.1%}')
print(f'mAP50-95 pontos : {m.pose.map:.1%}')

from IPython.display import Image, display
display(Image(filename='/content/runs/vigia/results.png', width=900))  # evolução ao longo das épocas

## 4. Ver as previsões nas imagens de validação (com a confiança de cada uma)

In [ ]:
import cv2, numpy as np
CORES = [(255, 100, 0), (0, 200, 255), (0, 0, 255)]  # traseira, meio, frente
imagens_val = sorted(glob.glob('/content/yolo/images/val/*.jpg'))
passo = max(1, len(imagens_val) // 8)
for caminho in imagens_val[::passo][:8]:
    r = melhor.predict(caminho, conf=0.25, verbose=False)[0]
    img = r.orig_img.copy()
    if len(r.boxes) == 0:
        print(caminho.split('/')[-1], '-> nenhum carro encontrado')
    for caixa, conf, kp, kc in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(),
                                   r.keypoints.xy.cpu().numpy(), r.keypoints.conf.cpu().numpy()):
        x1, y1, x2, y2 = caixa.astype(int)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 220, 0), 4)
        cv2.putText(img, f'carro {conf:.0%}', (x1, max(40, y1 - 15)), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 220, 0), 4)
        p = kp.astype(int)
        cv2.line(img, tuple(p[0]), tuple(p[1]), (0, 220, 0), 4)
        cv2.arrowedLine(img, tuple(p[1]), tuple(p[2]), (0, 220, 0), 4, tipLength=0.3)
        for ponto, cor in zip(p, CORES):
            cv2.circle(img, tuple(ponto), 12, cor, -1)
        print(caminho.split('/')[-1], f'-> carro {conf:.0%} | traseira {kc[0]:.0%} meio {kc[1]:.0%} frente {kc[2]:.0%}')
    _, png = cv2.imencode('.jpg', img)
    display(Image(data=png.tobytes(), width=360))

## 5. Baixar o modelo

Salve em `modelos/vigia-pose.pt` no projeto.

In [ ]:
import shutil
shutil.copy('/content/runs/vigia/weights/best.pt', '/content/vigia-pose.pt')
files.download('/content/vigia-pose.pt')

## 6. (Opcional) Testar num vídeo aqui mesmo no Colab

Envie um dos vídeos da pasta `Video/` e veja o resultado anotado + a confiança média.

In [ ]:
enviado = files.upload()  # escolha um .mp4
video = next(iter(enviado))
cap = cv2.VideoCapture(video)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
total = com_carro = 0
confs, confs_pts = [], []
gravador = None
for r in melhor.predict(video, conf=0.5, stream=True, verbose=False):
    total += 1
    if len(r.boxes):
        com_carro += 1
        confs += r.boxes.conf.cpu().tolist()
        confs_pts += r.keypoints.conf.cpu().tolist()
    quadro = r.plot()
    if gravador is None:
        h, w = quadro.shape[:2]
        gravador = cv2.VideoWriter('/content/anotado.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    gravador.write(quadro)
gravador.release()
print(f'Carro encontrado em {com_carro} de {total} frames ({com_carro / max(1, total):.0%})')
if confs:
    mp = np.mean(confs_pts, axis=0)
    print(f'Confiança média da detecção: {np.mean(confs):.0%}')
    print(f'Confiança média dos pontos: traseira {mp[0]:.0%} | meio {mp[1]:.0%} | frente {mp[2]:.0%}')
files.download('/content/anotado.mp4')